In [1]:
# @title 1. Install Dependencies (Fixed for Binary Compatibility)

import os
os.system("pip uninstall -y numpy")

os.system("pip install numpy==1.26.4")
!pip install -q --no-cache-dir \
    torch==2.5.1 \
    torchvision==0.20.1 \
    transformers==4.40.2 \
    pydantic==2.12.2 \
    matchms==0.27.0 \
    rdkit==2024.9.6 \
    simsimd==6.5.12 \
    hnswlib==0.8.0 \
    huggingface_hub

In [3]:
# @title 2. Hugging Face Login
# @markdown **Required:** Please enter your Hugging Face Access Token (Read permissions) to access the private model and database.

from huggingface_hub import login

login()

In [4]:
# @title 3. Download Source Code & Setup Environment
# @markdown Downloads `model_finetune.py` and `modular_csuep.py` from your Space.

import os
import shutil
from huggingface_hub import hf_hub_download

SPACE_ID = "Tingxie/CSU-EP"
source_files = ["model_finetune.py", "modular_csuep.py"]
print("⬇️ Downloading source code...")
for filename in source_files:
  try:
    cached_path = hf_hub_download(repo_id=SPACE_ID, filename=filename, repo_type="space")
    shutil.copy(cached_path, filename)
    print(f" - {filename} downloaded.")
  except Exception as e:
    print(f" ❌ Error downloading {filename}: {e}")
try:
  from model_finetune import CSUEP_finetune
  from modular_csuep import CsuepConfig
  print("✅ Custom modules imported successfully.")
except ImportError as e:
  print(f"❌ Import failed: {e}. Please check if the files exist in your Space.")

⬇️ Downloading source code...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


 - model_finetune.py downloaded.
 - modular_csuep.py downloaded.


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

✅ Custom modules imported successfully.


In [6]:
# @title 4. Load Model & Run Search
# @markdown Performs spectrum embedding and searches against the HNSW index.

import torch
import torch.nn.functional as F
import numpy as np
import sqlite3
import hnswlib
import pickle
import pandas as pd
from tqdm import tqdm
from matchms.importing import load_from_mgf, load_from_msp
import matchms.filtering as msfilters
from huggingface_hub import hf_hub_download

MODEL_REPO_ID = "Tingxie/CSU-EP"
DATA_REPO_ID = "Tingxie/CSU-EP-DB"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚙️ Using device: {device}")

def spectrum_processing(s):
    if s is None: return None
    s = msfilters.normalize_intensities(s)
    s = msfilters.select_by_mz(s, mz_from=0, mz_to=1000)
    s = msfilters.select_by_intensity(s, intensity_from=0.001)
    s = msfilters.require_minimum_number_of_peaks(s, n_required=2)
    return s

def MS2Embedding(spectrum, model):
    word_list = list(np.linspace(0, 1001, 1001, endpoint=False))
    word_list = [str(i) for i in word_list]
    word2idx = {'[PAD]': 1002, '[MASK]': 1003}
    for i, w in enumerate(word_list):
        word2idx[w] = i + 1
    spectrum = spectrum_processing(spectrum)

    spec_mz = spectrum.peaks.mz
    spec_intens = spectrum.peaks.intensities
    # Tokenization
    input_ids = []
    valid_indices = []
    for idx, s_val in enumerate(spec_mz):
        key = str(float(int(s_val)))
        if key in word2idx:
            input_ids.append(word2idx[key])
            valid_indices.append(idx)

    input_ids = np.array(input_ids)
    spec_intens = spec_intens[valid_indices]
    attention_mask = np.ones_like(input_ids)

    input_ids = torch.from_numpy(input_ids).long().to(device)
    intensities = torch.from_numpy(spec_intens).float().to(device)
    attention_mask = torch.from_numpy(attention_mask).long().to(device)

    with torch.no_grad():
        outputs = model.text_encoder(
            input_ids=input_ids.unsqueeze(0),
            intensities=intensities.unsqueeze(0),
            attention_mask=attention_mask.unsqueeze(0),
            return_dict=True,
            output_attentions=True
        )
        output_feats = outputs.last_hidden_state
        output_aggr_feats = model.pooler(output_feats, attention_mask.unsqueeze(0))
        output_aggr_feats = model.proj(output_aggr_feats)
        spectrum_embeddings = F.normalize(output_aggr_feats, dim=1)

    return spectrum_embeddings.detach().cpu().numpy()

print("⬇️ Downloading model")
model_path = hf_hub_download(repo_id=MODEL_REPO_ID, filename="checkpoints/model.pth", repo_type="space")
state_dict = torch.load(model_path, map_location=device)

model = CSUEP_finetune()
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print("✅ Model loaded successfully.")


print("⬇️ Downloading Database and HNSW Index (This might take a moment)...")
# Database
db_file = hf_hub_download(repo_id=DATA_REPO_ID, filename="csu-ep.db", repo_type="dataset")
database_conn = sqlite3.connect(db_file, check_same_thread=False)
cur = database_conn.cursor()
table_name = "CsuepDB"

index_file = hf_hub_download(repo_id=DATA_REPO_ID, filename="references_index.bin", repo_type="dataset")
p = hnswlib.Index(space='l2', dim=768)
p.load_index(index_file)
print("✅ Database and Index loaded.")

print("⬇️ Downloading Test Spectrum...")
query_file = hf_hub_download(repo_id=MODEL_REPO_ID, filename="test.msp", repo_type="space")
query_spectrum = list(load_from_msp(query_file))[0]
query_emb = MS2Embedding(query_spectrum, model).astype('float32')

print("🔍 Searching database...")
k = 10
labels, distances = p.knn_query(query_emb, k=k)
target_labels, target_distances, score_mode = labels[0], distances[0], "hnsw"

results = []
seen_inchikeys = set()

for idx, dist in zip(target_labels, target_distances):
    cur.execute(f"SELECT SMILES, InChIKey, ShortInChIKey, MonoisotopicMass, PredictedSpectrum FROM {table_name} WHERE id=?;", (int(idx),))
    row = cur.fetchone()
    if not row: continue
    inchikey = row[1]
    if inchikey in seen_inchikeys: continue
    seen_inchikeys.add(inchikey)

    cos_sim = 1 - dist/2 if score_mode == "hnsw" else 1 - dist

    results.append({
        "Rank": 0, # Will update after sorting
        "SMILES": row[0],
        "InChIKey": row[1],
        "ShortInChIKey": row[2],
        "Mass": round(row[3], 4),
        "Score": round(float(cos_sim), 4)
    })

results.sort(key=lambda x: x["Score"], reverse=True)
for n, item in enumerate(results, 1):
    item["Rank"] = n

df_results = pd.DataFrame(results)
print("\n🏆 Top Identification Results:")
print(df_results[['Rank', 'Score', 'Mass', 'SMILES', 'InChIKey']].head(10))
database_conn.close()

⚙️ Using device: cpu
⬇️ Downloading model


/tmp/ipython-input-1162561183.py:73: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location=device)


✅ Model loaded successfully.
⬇️ Downloading Database and HNSW Index (This might take a moment)...
✅ Database and Index loaded.
⬇️ Downloading Test Spectrum...
2026-01-14 11:14:31,365:WARNING:matchms:add_precursor_mz:No precursor_mz found in metadata.


🔍 Searching database...

🏆 Top Identification Results:
   Rank   Score      Mass                         SMILES  \
0     1  0.8968  204.0011              CN(C)c1nc(Br)nn1C   
1     2  0.8728  203.9898              Cc1onc(N(C)C)c1Br   
2     3  0.7506  206.1531      Cc1cc(N2CCC(N)CC2)nc(C)n1   
3     4  0.7457  206.1280     CN/C(=N\C#N)NCCCc1c[nH]cn1   
4     5  0.7325  206.0916  C#CCN(C)/N=N/c1nc[nH]c1C(N)=O   
5     6  0.7166  206.1168       NC1CCN(C(=O)c2cnccn2)CC1   
6     7  0.7156  206.1531       CCCn1ncc(C2=CCCN(C)C2)n1   
7     8  0.7096  206.1280          CCN(CC)c1ncnc2c1nnn2C   
8     9  0.7085  206.1531      Cc1cc(N2CCCC(N)C2)nc(C)n1   
9    10  0.7072  206.1531        CNc1cc(C2CCCNC2)nc(C)n1   

                      InChIKey  
0  QPGCLRNPGBADMD-UHFFFAOYSA-N  
1  IFNAXMAJTXIRLM-UHFFFAOYSA-N  
2  WZVIPIYHPMSOHK-UHFFFAOYSA-N  
3  CXTGREPICTYOGB-UHFFFAOYSA-N  
4  JGRODHQKLZFMNE-OUKQBFOZSA-N  
5  OOPVDYMIOCWMPK-UHFFFAOYSA-N  
6  ZYLDIOUZTVCBBN-UHFFFAOYSA-N  
7  FCDVBLAOLPSXFM-UH